# Radiomics Feature Extraction

This notebook extracts PyRadiomics features from the largest lesion component. SimpleITK is used to preserve image spacing and orientation metadata.

In [ ]:
from pathlib import Path
import pandas as pd

from src.radiomics_extraction import extract_features_from_manifest

In [ ]:
MANIFEST_CSV = Path('data/manifest.csv')
CONFIG_YAML = Path('configs/pyradiomics_original_features.yaml')
OUTPUT_DIR = Path('data/features')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_CSV)

In [ ]:
raw_features, raw_failures = extract_features_from_manifest(
    manifest=manifest,
    image_column='image_path',
    label_column='label_path',
    case_id_column='case_id',
    output_csv=OUTPUT_DIR / 'raw_radiomic_features.csv',
    config_path=CONFIG_YAML,
    lesion_label=1,
)
raw_failures.to_csv(OUTPUT_DIR / 'raw_radiomics_failures.csv', index=False)
raw_features.head()

After running notebook 01, merge `normalized_image_path` from `results/dentin_normalization/dentin_normalization_stats.csv` into the manifest, then run the normalized extraction.

In [ ]:
norm_stats = pd.read_csv('results/dentin_normalization/dentin_normalization_stats.csv')
manifest_norm = manifest.merge(
    norm_stats[['case_id', 'normalized_image_path']],
    on='case_id',
    how='inner',
)

normalized_features, normalized_failures = extract_features_from_manifest(
    manifest=manifest_norm,
    image_column='normalized_image_path',
    label_column='label_path',
    case_id_column='case_id',
    output_csv=OUTPUT_DIR / 'dentin_normalized_radiomic_features.csv',
    config_path=CONFIG_YAML,
    lesion_label=1,
)
normalized_failures.to_csv(OUTPUT_DIR / 'dentin_normalized_radiomics_failures.csv', index=False)
normalized_features.head()